# Battle 通用远程 GPU Worker（单 cell 极简版）

**用法：填好唯一代码 cell 顶部的 `hub_url` / `hub_token`（push 模式再加 `push_token`），Run 它，其余全自动。**

| 自动步骤 | 说明 |
|---|---|
| 保活 | Colab 每 60s 模拟点击 · Kaggle 每 120s 心跳（代码引导等待期也保活） |
| 代码引导 | 从 hub `GET /code` 拉 code.zip；hub 还没发布过 job 时**每 30s 等待重试**（先起 worker 后起 trainer 是合法顺序） |
| 运行时接管 | `remote/notebook_runtime.py`（随 code.zip 下发）：设备探测（CUDA/TPU/CPU，TPU 内核不碰设备）、worker 监督（热替换 86 自动重启）、崩溃退避重启、空闲自动退出 |
| 多卡 | 双/多卡默认启用 DataParallel（PPO/BC 均生效）；与历史单卡 run **不可逐位比**——要单卡把 `use_multi_gpu` 改 `False`（opt-out） |
| 任务类型 | PPO（kind=ppo）与 BC（kind=bc）同一 worker，manifest 自带全部上下文 |

**运行时逻辑住在 code.zip 里**——修运行时不用重发 notebook，重跑本 cell 即拉到新版。

**连接信息**：`hub_url` = 控制台 cloudflared 卡片隧道 URL（= `rl.remote_hubs.<课程>`）；
`hub_token` = 训练机 `nn-training/rl-config.json` 的 `rl.remote_token`。多课程并行每课一隧道。

**停止**：中断本 cell（■）= 干净停机。**配额**：Kaggle 30h GPU/周、单次 9h（`max_session_hours` 调小省配额）。

### 常见问题

| 症状 | 处理 |
|---|---|
| `FATAL: /ping HTTP 401` | hub_token 与训练机 `rl.remote_token` 不一致 |
| `FATAL: hub 不可达` | 隧道过期——控制台重启 cloudflared 后更新 URL |
| 等待 code.zip 超 1h | hub 侧从未发布过任何 job——确认训练机 trainer/BC 编排器已启动 |
| `TPU ... Device or resource busy` | TPU 独占设备被持有；本内核占用只能 Runtime → Restart session（探测输出里有占用者诊断） |
| payload 校验失败 | 传输损坏，自动重试；hub 幂等保证不重复训练 |


In [ ]:
# @title 一键连接 hub —— 填参数 → Run 本 cell，其余全自动
# ============================================================================
# Battle 通用远程 GPU Worker（单 cell 极简版，2026-09-13）
#
# 运行时逻辑（设备探测 / pull / push / 崩溃退避重启）全部住在 hub 的 code.zip 里
# （nn-training/remote/notebook_runtime.py）——修运行时不用重发 notebook，重跑本
# cell 即拉到新版；job 级代码变更照旧热替换。
#
# 用法：改 ⚙️ 连接参数 → Run 本 cell。自动流程：
#   保活（Colab 每 60s 模拟点击 / Kaggle 每 120s 心跳——等待期也保活）
#   → GET /code 引导（未就绪时每 30s 等待重试：先起 worker 后起 trainer 合法）
#   → remote.notebook_runtime.run_notebook 接管（设备探测、worker 监督、
#     热替换 86 重启、崩溃退避重启、空闲自动退出）
# 停止 = 中断本 cell（■，子进程全回收）。任务类型：PPO 与 BC 同 worker，无需区分。
# 连接信息：hub_url = 控制台 cloudflared 卡片隧道 URL（rl.remote_hubs.<课程>）；
#           hub_token = 训练机 rl-config 的 rl.remote_token。多课每课一隧道，别串线。
# ============================================================================
import os
import sys
import threading
import time

CFG = {
    # ── ⚙️ 连接参数（通常只改这三行）──
    "mode": "pull",            # "pull"（worker 轮询 hub，推荐）| "push"（hub 推过来）
    "hub_url": "https://your-tunnel.trycloudflare.com",
    "hub_token": "YOUR_TOKEN_HERE",
    # ── push 模式专用 ──
    "push_port": 8790,
    "push_token": "YOUR_TOKEN_HERE",
    "cloudflared_path": "",    # 留空 = 自动查找/安装
    # ── 行为参数（一般不用动）──
    "device": "auto",          # auto | cuda | cuda-dp | tpu | cpu
    "use_multi_gpu": True,     # 多卡默认启用 DataParallel（PPO 1.92×@T4x2 实测；BC 同支持）。
                               # 与历史单卡 run 不可逐位比——要逐位可比改 False（opt-out）。
    "max_session_hours": 9,    # Kaggle 单次最长 9h；短腿课程调小省配额
    "poll_interval_sec": 5,
    "idle_floor_sec": 3600,    # worker 空闲退出下限（实际 = max(此值, (会话-1)h)）
    "max_worker_restarts": 5,  # 崩溃退避重启上限
}


def _log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] [battle-rl] {msg}", flush=True)


# ── 保活先行：/code 等待期也可能很长（Colab 免费 ~90min 空闲回收）──
_keepalive_stop = threading.Event()


def _keepalive_loop() -> None:
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
        while not _keepalive_stop.is_set():
            try:
                from IPython.display import Javascript, display

                display(
                    Javascript(
                        "function f(){document.querySelector("
                        '"colab-connect-button")?.click();}setTimeout(f,1000);'
                    )
                )
            except Exception:
                pass
            _keepalive_stop.wait(60)
    else:
        n = 0
        while not _keepalive_stop.is_set():
            print(
                f"[{time.strftime('%H:%M:%S')}] [battle-rl] [keepalive] alive ({n * 2} min)",
                flush=True,
            )
            n += 1
            _keepalive_stop.wait(120)


threading.Thread(target=_keepalive_loop, daemon=True, name="keepalive").start()
_log("Keepalive 已启动")


# ── 代码引导：GET /code → 解包 → sys.path[0]（唯一必须留在 cell 里的逻辑）──
import hashlib
import io
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

_deadline = time.time() + 3600
_attempt = 0
while True:
    _attempt += 1
    try:
        _req = urllib.request.Request(
            CFG["hub_url"].rstrip("/") + "/code",
            headers={"Authorization": "Bearer " + str(CFG["hub_token"])},
        )
        with urllib.request.urlopen(_req, timeout=120) as _resp:
            _raw = _resp.read()
        _log(
            f"code.zip 就绪: {len(_raw)} bytes, "
            f"sha256={hashlib.sha256(_raw).hexdigest()[:12]}…（第 {_attempt} 次尝试）"
        )
        break
    except urllib.error.HTTPError as _e:
        if _e.code in (401, 403):
            raise SystemExit(
                f"[battle-rl] FATAL: /code HTTP {_e.code} — hub_token 与训练机 rl.remote_token 不一致"
            ) from None
        if _e.code != 404:
            _log(f"/code HTTP {_e.code} —— 30s 后重试")
        elif time.time() > _deadline:
            raise SystemExit(
                "[battle-rl] FATAL: 等待 code.zip 超 1h —— 训练机从未发布过 job，先启动 trainer/BC 编排器"
            ) from None
        elif _attempt == 1 or _attempt % 6 == 0:
            _log("hub 尚无 code.zip（训练机还没发布过 job）——每 30s 重试；先起 worker 后起 trainer 是合法顺序…")
        time.sleep(30)
    except Exception as _e:
        _log(f"hub 连接异常（{type(_e).__name__}: {_e}）—— 30s 后重试")
        time.sleep(30)

_code_dir = Path("/tmp/worker-code")
_code_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(_raw)) as _z:
    _z.extractall(_code_dir)
sys.path.insert(0, str(_code_dir))
_log("code.zip 解包就绪——移交 remote.notebook_runtime（设备探测/worker 监督/崩溃退避）")

from remote.notebook_runtime import run_notebook

CFG["keepalive_stop"] = _keepalive_stop
CFG["log"] = _log
CFG["code_dir"] = str(_code_dir)
run_notebook(CFG)
